In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import logging
import os
from IPython.display import display
from dataclasses import dataclass

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser
from gsm_benchmarker.results_analyser.prompt_effect_analyser import PromptEffectAnalyser
from gsm_benchmarker.results_analyser.prompt_result import PromptResult
from gsm_benchmarker.results_analyser.plotting_utils import plot_glmm, plot_acc_change_distribution, Colour, plot_prompt_format_comparison

logger = logging.getLogger('notebook')

plt.style.use('default')
plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
METRIC = "correct"

In [ ]:
MARGIN = 0.1

ALPHA = 0.05
ALPHA_THRESHOLD = (1 + MARGIN) * ALPHA
PROJECTED_ALPHA = 0.2
PROJECTED_ALPHA_THRESHOLD = (1 + MARGIN) * PROJECTED_ALPHA

In [ ]:
OUTPUTS = Path("./outputs").resolve()
os.makedirs(OUTPUTS, exist_ok=True)
OUTPUTS_FOLDER = str(OUTPUTS) + "/"

In [ ]:
result_kwargs = dict(metric=METRIC, notebook=True, save_dest=OUTPUTS)
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()


pilot_gsm_result = PromptResult(
    pp / "mini_20x50x4__14_11/final",
    colour=Colour('green'),
    full_label="GSM prompt (pilot study)",
    short_label="GSM_pilot",
    **result_kwargs
)

gsm_result = PromptResult(
    pp / "default_full__06_02/final",
    colour=Colour('green'),
    full_label="GSM prompt",
    **result_kwargs
)

nonformal_result = PromptResult(
    pp / 'nonformalised_full__20_04/final',
    colour=Colour("lightseagreen"),
    full_label="Non-formalised NL prompt",
    short_label="nonformal",
    baseline=gsm_result.mres,
    **result_kwargs
)

formal_result = PromptResult(
    pp / 'formalised_full__16_04/final',
    colour=Colour("royalblue"),
    full_label="Formalised NL prompt",
    short_label="formal",
    baseline=gsm_result.mres,
    **result_kwargs
)

code_result = PromptResult(
    pp / 'code_full__09_02/corrected',
    colour=Colour("slateblue"),
    full_label="Code output prompt",
    short_label="code",
    baseline=gsm_result.mres,
    **result_kwargs
)

full_results = {
    'gsm': gsm_result,
    'nonformal': nonformal_result,
    'formal': formal_result,
    'code': code_result
}


### Modelling question difficulty

In [ ]:
gsm_result.mres.plot_question_difficulty_per_model(
    # title="Leave-one-(model-)out question difficulty",
    save_prefix=OUTPUTS_FOLDER,
)

In [ ]:
fig_qd_hist = gsm_result.mres.plot_question_difficulty_histogram(
    n_levels=5,
    color=gsm_result.colour.value,
    cumulative_color=gsm_result.colour.darken(factor=0.6),
    save_prefix=OUTPUTS_FOLDER,
)
display(fig_qd_hist)

In [ ]:
gsm_result.mres.plot_number_counts(save_prefix=OUTPUTS_FOLDER)

## Question 1
Are the accuracy drops reported in the GSM-Symbolic paper actually significant?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with GSM-Symbolic prompt.

### 1A: Pilot run
Checking on a subset of the benchmark first. Projecting results to the full dataset with alpha = 20%.

50/100 questions, 20/50 template variations -> 1000/5000 total questions in the 'main variant' (20% of the benchmark) (+ 50 questions in the original GSM8K variant)

In [ ]:
pilot_gsm_result.plot_variant_effect(projected_alpha=PROJECTED_ALPHA)

#### Candidate models
Models which, based on the pilot results, are worth checking on the full dataset.

In [ ]:
candidate_models = pilot_gsm_result.get_significant_models(alpha=PROJECTED_ALPHA_THRESHOLD)
candidate_models

In [ ]:
gsm_result.models = candidate_models

### 1B: Full benchmark
Re-running the tests on the full benchmark (100 + 5000 questions) with the identified candidate models.

In [ ]:
gsm_result.plot_variant_effect()

In [ ]:
significant_models = gsm_result.get_significant_models(alpha=ALPHA_THRESHOLD)
significant_models

In [ ]:
for res in (nonformal_result, formal_result, code_result):
    res.models = significant_models


## Question 2
Do alternative prompt formats remove the variant dependency?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with code prompt and formalised natural language prompt.

In [ ]:
nonformal_result.plot_variant_effect()

In [ ]:
formal_result.plot_variant_effect()

In [ ]:
code_result.plot_variant_effect()

### Alternative prompts evaluation
Do alternative prompt formats result in significant performance changes?

Evaluating performance on 'main' variant with the alternative prompts vs GSM-Symbolic prompt.

In [ ]:
nonformal_result.plot_prompt_effect()

In [ ]:
formal_result.plot_prompt_effect()

In [ ]:
code_result.plot_prompt_effect()

In [ ]:
code_result.plot_acc_change_dist()

In [ ]:
code_result.summary()

### Mean accuracy by prompt format on `main` (significant models)

In [ ]:
prompt_order = list(full_results.keys())
prompt_labels = {
    'standard': 'GSM prompt',
    'code': 'Code prompt',
    'formal': 'Formalised NL prompt',
    'nonformal': 'Non-formalised NL prompt',
}
prompt_colours = {key: full_results[key].colour.value for key in prompt_order}
prompt_mres = {
    'standard': mres_standard,
    'code': mres_code,
    'formal': mres_formal,
    'nonformal': mres_nonformal,
}
acc_records = []
prompt_effect_records = []
variant_records = []
existing_variant_dfs = {
    'standard': variant_effect_df_full,
    'code': variant_effect_df_code,
    'formal': variant_effect_df_formal,
    'nonformal': variant_effect_df_nonformal,
}
existing_prompt_effect_dfs = {
    'code': acc_change_df_code,
    'formal': acc_change_df_formal,
    'nonformal': acc_change_df_nonformal,
}
for prompt_key in prompt_order:
    mres = prompt_mres[prompt_key]
    main_acc_per_model = (
        mres
        .variants['main']
        .get_accuracies_per_model_and_template_id(metric=METRIC)
        .groupby('model')
        .mean()
        .reindex(significant_models)
    )
    gsm8k_acc_per_model = (
        mres
        .variants['GSM8K']
        .get_accuracies_per_model_and_template_id(metric=METRIC)
        .groupby('model')
        .mean()
        .reindex(significant_models)
    )
    for model in significant_models:
        main_acc = main_acc_per_model.loc[model]
        gsm8k_acc = gsm8k_acc_per_model.loc[model]
        acc_records.append({
            'model': model,
            'prompt': prompt_key,
            'mean_accuracy': float(main_acc) if pd.notna(main_acc) else float('nan'),
            'gsm8k_mean_accuracy': float(gsm8k_acc) if pd.notna(gsm8k_acc) else float('nan'),
        })
    variant_df_prompt = existing_variant_dfs[prompt_key]
    for model in significant_models:
        if model in variant_df_prompt.index:
            variant_records.append({
                'model': model,
                'prompt': prompt_key,
                'accuracy_change': float(variant_df_prompt.loc[model, 'mean_diff']),
                'p_value': float(variant_df_prompt.loc[model, 'p_value']),
            })
    if prompt_key == 'standard':
        for model in significant_models:
            prompt_effect_records.append({
                'model': model,
                'prompt': prompt_key,
                'prompt_accuracy_change': float('nan'),
                'prompt_p_value': float('nan'),
            })
    else:
        prompt_effect_df_prompt = existing_prompt_effect_dfs[prompt_key]
        for model in significant_models:
            if model in prompt_effect_df_prompt.index:
                prompt_effect_records.append({
                    'model': model,
                    'prompt': prompt_key,
                    'prompt_accuracy_change': float(prompt_effect_df_prompt.loc[model, 'mean_diff']),
                    'prompt_p_value': float(prompt_effect_df_prompt.loc[model, 'p_value']),
                })
acc_df = pd.DataFrame(acc_records)
prompt_effect_df = pd.DataFrame(prompt_effect_records)
variant_df = pd.DataFrame(variant_records)
plot_df = (
    acc_df
    .merge(prompt_effect_df, on=['model', 'prompt'], how='left')
    .merge(variant_df, on=['model', 'prompt'], how='left')
)
fig = plot_prompt_format_comparison(
    plot_df,
    selected_models=significant_models,
    prompt_order=prompt_order,
    prompt_labels=prompt_labels,
    prompt_colours=prompt_colours,
    mean_ylabel='Mean accuracy on main',
    gsm8k_mean_ylabel='Mean accuracy on GSM8K',
    prompt_effect_ylabel='Prompt performance delta',
    variant_ylabel='Symbolic performance delta',
    include_gsm8k_mean=True,
    save_prefix=OUTPUTS_FOLDER,
)
display(fig)


### GSM8K vs main mean accuracy by model


In [ ]:
all_prompts_summary = pd.concat([r.summary() for r in full_results.values()], keys=[r.short_label for r in full_results.values()], names=['prompt', 'quantity'])
all_prompts_summary

In [ ]:
n_models = len(significant_models)
n_cols = 2
n_rows = n_models // n_cols + n_models % n_cols

x_data = all_prompts_summary.xs('GSM8K_acc', level='quantity')
y_data = all_prompts_summary.xs('main_acc', level='quantity')
sig_data = all_prompts_summary.xs('delta_symb_significant', level='quantity')
colours = {r.short_label: r.colour.value for r in full_results.values()}

fig, axes = plt.subplots(n_rows, n_cols, sharex='all', sharey='all', figsize=(10, 8))
for i, (ax, model) in enumerate(zip(axes.flatten(), significant_models)):
    ax.set_title(model)
    ax.set_xlabel("Mean accuracy on GSM8K")
    ax.set_ylabel("Symbolic performance delta ($\Delta_{symb}$)")
    model_data = pd.concat([x_data[model], y_data[model], sig_data[model]], axis=1, keys=['x', 'y', 'significant'])

    for prompt in model_data.index:
        x_val, y_val, _ = model_data.loc[prompt]
        colour = colours[prompt]
        ax.plot(x_val, y_val, marker='o', c=colour, label=prompt)
        ax.annotate(prompt, (x_val, y_val), textcoords='offset points', xytext=(4, 4), fontsize=8, color=colour)

    for pair in (['GSM', 'nonformal'], ['nonformal', 'formal'], ['formal', 'code']):
        pair_data = model_data.loc[pair]
        ax.plot(pair_data['x'], pair_data['y'], lw=0.5, c='darkslategrey')

    model_sig_data = model_data[model_data.significant]
    if sig_data.size:
        ax.plot(model_sig_data['x'], model_sig_data['y'], marker='o', lw=0, c='none', mec='darkred', ms=12, label='significant $\Delta_{symb}$')


handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title='Prompt / significance', loc='lower center', ncol=6, frameon=True)
fig.tight_layout(rect=(0, .05, 1, 1))


display(fig)